# Using ``rename_epw_files()`` to rename the EPWs for proper data analysis after simulation

``rename_epw_files`` renames your EPW files following the naming convention "Country_City_RCPscenario-Year". It takes the Country and City fields from the coordinates of the EPW file or from dictionaries entered by the user, and the RCPscenario and Year fields from the original name. If there is no reference to these in the original name, it considers the EPW file to be for the Present scenario.

To get the address from the coordinates, ``rename_epw_files`` uses OpenStreetMap, so it needs an internet connection.

In [1]:
from accim.data.preprocessing import rename_epw_files
help(rename_epw_files)

Help on class rename_epw_files in module accim.data.preprocessing:

class rename_epw_files(builtins.object)
 |  rename_epw_files(
 |      filelist=None,
 |      rename_city_dict: dict = None,
 |      country_to_city_dict: dict = None,
 |      confirm_renaming=None,
 |      confirm_deletion=None
 |  )
 |
 |  Renames the EPW files following the name convention 'Country_City_RCPscenario-Year'.
 |  The Country and City fields are computed based on the coordinates of the EPW,
 |  and the RCPscenario and Year are taken from the original name.
 |  If no reference is found, the EPWs are considered to be for Present scenario.
 |
 |  :param filelist: A list of the EPW files.
 |      If omitted, it will rename all EPWs in that folder.
 |  :type filelist: list
 |  :param rename_city_dict: A dict to set the city field for each EPW file.
 |      It must follow the pattern
 |      {'city name to be search in epw file name': 'city name for the epw file if found`}
 |  :type rename_city_dict: dict
 |  :

First of all, let's see what files we do have in the folder:

In [2]:
import os
previous_files = [i for i in os.listdir()]
previous_files

['GBR_Aberdeen.Dyce.030910_IWEC.epw',
 'GBR_London.Gatwick.037760_IWEC.epw',
 'RCP26_2100_GC03_Ponta_Grossa.epw',
 'using_rename_epw_files.ipynb',
 '__init__.py']

You can see there are 3 EPW files, which are:

In [3]:
old_epws = [i for i in os.listdir() if i.endswith('.epw')]
print(old_epws)

['GBR_Aberdeen.Dyce.030910_IWEC.epw', 'GBR_London.Gatwick.037760_IWEC.epw', 'RCP26_2100_GC03_Ponta_Grossa.epw']


So let's rename them. ``rename_epw_files`` asks some questions; in this notebook, the answers are shown after each question. First, it shows the proposed names and asks for the IDs of the files whose city needs some amendment. Then, for each of those IDs, it shows the address obtained from the coordinates and asks for the new city name. Finally, it asks for the IDs of the files that should not be renamed, and for confirmation. Since we pass ``confirm_deletion=False``, the original files are kept.

The names obtained from the coordinates of the first two files include the airports where the weather stations are (Aberdeen-Dyce and London-Gatwick), so we enter **0 1** to amend them, and then **Aberdeen** and **London**. We hit enter to exclude none of the files, and confirm with **y**.

## Using data from coordinates

In [4]:
from accim.data.preprocessing import rename_epw_files
rename_epw_files(
    confirm_deletion=False,
)

Since no match has been found between RCP or SSP scenarios and EPW file name, Present scenario has been assigned to the following EPW files:
GBR_Aberdeen.Dyce.030910_IWEC.epw
GBR_London.Gatwick.037760_IWEC.epw
Since no match has been found between RCP or SSP scenario Year and EPW file name, Present year has been assigned to the following EPW files:
GBR_Aberdeen.Dyce.030910_IWEC.epw
GBR_London.Gatwick.037760_IWEC.epw
The geolocation process has taken: 1.52 seconds (0.51 s/EPW)

The previous and new names of the EPW files and their unique IDs are:
ID: 0 / GBR_Aberdeen.Dyce.030910_IWEC / United-Kingdom_Aberdeen-Dyce_Present
ID: 1 / GBR_London.Gatwick.037760_IWEC / United-Kingdom_London-Gatwick_Present
ID: 2 / RCP26_2100_GC03_Ponta_Grossa / Brazil_Ponta-Grossa_RCP26-2100

If any of the city or subcountry names needs some amendment (if you are not happy with any of the available options, you can exclude it from renaming at the next stage), please enter the EPW IDs separated by space; otherw

You can see above that there was no reference to RCP scenarios in the original name of 2 of the EPW files, so these have been considered as Present scenario. The same applies to the Year field. Finally, ``rename_epw_files`` reports the previous and the new names of the EPW files.

We can see the new EPWs are:

In [5]:
new_epws = [i for i in os.listdir() if not(any(i in j for j in old_epws)) and i.endswith('.epw')]
print(new_epws)

['Brazil_Ponta-Grossa_RCP26-2100.epw', 'United-Kingdom_Aberdeen_Present.epw', 'United-Kingdom_London_Present.epw']


Let's delete the new files so that we can run the notebook again.

In [6]:
for i in new_epws:
    os.remove(i)

## Using data from user-defined dicts

Sometimes, the address obtained from the coordinates is not what you expect, or OpenStreetMap cannot be reached. Therefore, there is also a way to rename the files using user-defined data in the shape of dictionaries. Firstly, to define the city or subcountry of the EPW files, we need to enter a dictionary such as the following:

In [7]:
rename_city_dict = {
    'Aberdeen': 'Aberdeen city',
    'London': 'London city'
}
rename_city_dict

{'Aberdeen': 'Aberdeen city', 'London': 'London city'}

If we enter that dictionary, accim will look for 'Aberdeen' in the EPW names and set 'Aberdeen city' in the city or subcountry field (written 'Aberdeen-City' in the new name). The same with 'London' and 'London city'.

To define which country is related to the city or subcountry, a different dictionary must be used, in the following shape:

In [8]:
country_to_city_dict = {
    'United kingdom': ['Aberdeen city', 'London city']
}
country_to_city_dict

{'United kingdom': ['Aberdeen city', 'London city']}

In this case, 'United kingdom' will be set as the country of the cities defined in the ``rename_city_dict`` argument, that is, 'Aberdeen city' and 'London city'.

If the city or the country of an EPW file is not defined in these dictionaries, accim takes it from the coordinates. In this case, 'RCP26_2100_GC03_Ponta_Grossa.epw' is not in the dictionaries, so its city and country will come from the coordinates. Likewise, if a city were not related to any country, as in the dictionary below, its country would come from the coordinates:

In [9]:
country_to_city_dict_not_entirely_defined = {
    'United kingdom': ['Aberdeen city']
}
country_to_city_dict_not_entirely_defined

{'United kingdom': ['Aberdeen city']}

In that case, we did not say that the country of 'London city' is 'United kingdom', so it would be taken from the coordinates. Let's rename the files with the dictionaries ``rename_city_dict`` and ``country_to_city_dict``. accim asks the same questions as before: this time, we hit enter twice (no amendments and no exclusions) and confirm with **y**.

In [10]:
from accim.data.preprocessing import rename_epw_files
rename_epw_files(
    rename_city_dict=rename_city_dict,
    country_to_city_dict=country_to_city_dict,
    confirm_deletion=False,
)

Since no match has been found between RCP or SSP scenarios and EPW file name, Present scenario has been assigned to the following EPW files:
GBR_Aberdeen.Dyce.030910_IWEC.epw
GBR_London.Gatwick.037760_IWEC.epw
Since no match has been found between RCP or SSP scenario Year and EPW file name, Present year has been assigned to the following EPW files:
GBR_Aberdeen.Dyce.030910_IWEC.epw
GBR_London.Gatwick.037760_IWEC.epw
The geolocation process has taken: 0.18 seconds (0.06 s/EPW)

The previous and new names of the EPW files and their unique IDs are:
ID: 0 / GBR_Aberdeen.Dyce.030910_IWEC / United-Kingdom_Aberdeen-City_Present
ID: 1 / GBR_London.Gatwick.037760_IWEC / United-Kingdom_London-City_Present
ID: 2 / RCP26_2100_GC03_Ponta_Grossa / Brazil_Ponta-Grossa_RCP26-2100

If any of the city or subcountry names needs some amendment (if you are not happy with any of the available options, you can exclude it from renaming at the next stage), please enter the EPW IDs separated by space; otherwise

We can see the new EPWs are:

In [11]:
new_epws = [i for i in os.listdir() if not(any(i in j for j in old_epws)) and i.endswith('.epw')]
print(new_epws)

['Brazil_Ponta-Grossa_RCP26-2100.epw', 'United-Kingdom_Aberdeen-City_Present.epw', 'United-Kingdom_London-City_Present.epw']


Let's delete the new files so that we can run the notebook again.

In [12]:
for i in new_epws:
    os.remove(i)